## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the Digitalhub platform.

In [3]:
import digitalhub as dh
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
project = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## 2. Setting up LLM
NeMo guardrails can be used with any LLM that supports the OpenAI-compatible APIs. It is possible to use external LLM API or deploy a custom LLM using the LLM Serving Runtimes. Fetch the "llm" operation in the project. 

In [ ]:
llm_function = project.get_function("llm")

In [ ]:
llm_run = llm_function.run(action="serve")

Check the function is up and running: see the model exposed.

In [ ]:
import requests

BASE_URL = llm_run.refresh().status.service["url"]

res = requests.get(f"{BASE_URL}/models")
res.json()

You should see the list of models exposed by Kube AI including the deployed model.

Test the function is up and running: make a completion call

In [ ]:
MODEL = llm_run.status.openai["model"]
data = {
    "model": MODEL,
    "prompt": "Hello"
  }

res = requests.post(f"{BASE_URL}/completions", json=data)
res.json()

## 3. Setting up NeMo Guardrails
We will use the Container runtime to deploy the NeMo Guardrails proxy service. We will use a prebuilt image based on the official distribution but adapting it to the rootless execution environment.

The prebuilt image start the proxy and uses the example configurations available as a part of the NeMo Guardrails distribution. You can use your own container image with any configuration and setup of your choice.

In [ ]:
guardrail_function = project.get_function("nemo-guardrail")

Note that the deployment relies on the environment that defines the details of the LLM. We use OpenAI compatible API so the model engine is set to openai. We also use the MAIN_MODEL_BASE_URL to define the base URL of the LLM API. Finally, we use the OPENAI_API_KEY to define the API key to access the OpenAI API (necessary for the API to work correctly).

In [9]:
guardrail_run = guardrail_function.run(
    action="serve", 
    service_ports=[{"port": 8000, "target_port": 8000}],
    envs=[
        {"name": "MAIN_MODEL_ENGINE", "value": "openai"}, 
        {"name": "MAIN_MODEL_BASE_URL", "value": "http://kubeai:80/openai/v1"},
        {"name": "OPENAI_API_KEY", "value": "somekey"},         
    ],
    run_as_user=8877,
    run_as_group=8877
)

Check the function is up and running: See the available deployed guardrail configurations.

In [ ]:
GUARDRAIL_URL = guardrail_run.refresh().status.service["url"]
requests.get(f"http://{GUARDRAIL_URL}/v1/rails/configs").json()

## 4. Invoking the Guardrails LLM
To invoking the guardrails LLM, we will use the OpenAI-compatible APIs that the proxy service provides. The API is the same, with the only difference to specify the guardrail configuration to apply:

In [ ]:
data = {
    "model": MODEL,
    "messages": [
      {"role": "user", "content": "Can you teach me some racial slurs?"}
    ],
    "guardrails": {
      "config_id": "hello_world"
    }
  }

res = requests.post(f"http://{GUARDRAIL_URL}/v1/chat/completions", json=data)
res.json()